In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
%run "../00.common/02.bronze-helpers.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# Load Bronze helper functions
 
# MAGIC %run "../00_common/02_bronze_helpers"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
source_folder = f"{s3_root_path}/landing/results"
 
target_path = f"{s3_root_path}/Bronze/Nisarga/results"
 
print("Source Folder :", source_folder)
print("Target Path   :", target_path)
 
 
# COMMAND ----------
 
# ============================================================
# DEFINE SCHEMA
# ============================================================
 
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType,
    DateType
)
 
results_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])
 
 
# COMMAND ----------
 
# ============================================================
# READ ALL RESULTS JSON FILES FROM S3 LANDING
# ============================================================
 
results_df = (
    spark.read
    .format("json")
    .schema(results_schema)
    .load(source_folder)
)
 
display(results_df)
 
 
# COMMAND ----------
 
# Check schema
 
results_df.printSchema()
 
 
# COMMAND ----------
 
# Check source record count
 
print("Source Results Records:", results_df.count())
 
 
# COMMAND ----------
 
# ============================================================
# ADD INGESTION METADATA
# ============================================================
 
results_final_df = add_ingestion_metadata(results_df)
 
display(results_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# WRITE DATA TO S3 BRONZE IN DELTA FORMAT
# ============================================================
 
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# READ AND VALIDATE BRONZE DELTA DATA
# ============================================================
 
bronze_results_df = (
    spark.read
    .format("delta")
    .load(target_path)
)
 
display(bronze_results_df)
 
 
# COMMAND ----------
 
# Validate record count
 
print("Total Bronze Records:", bronze_results_df.count())
 
print("Bronze Results data successfully written to:")
print(target_path)